# 🚗 VehiclEye — Entrenamiento en Google Colab

**Duración:** ~2 horas con GPU T4 gratis  
**Ejecuta cada celda con Shift+Enter y espera ✅ antes de continuar.**

## CELDA 1 — Verificar GPU y configurar entorno

In [ ]:
# Silenciar DeprecationWarnings de jupyter_client (no afectan el código)
import warnings
warnings.filterwarnings('ignore')

import os, sys, torch

if torch.cuda.is_available():
    gpu = torch.cuda.get_device_name(0)
    mem = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f'✅ GPU: {gpu}  ({mem:.1f} GB VRAM)')
else:
    print('⚠️  Sin GPU — ve a: Entorno de ejecución → Cambiar tipo → GPU T4')

print(f'✅ Python {sys.version[:6]}  |  PyTorch {torch.__version__}')

## CELDA 2 — Instalar dependencias

In [ ]:
import warnings; warnings.filterwarnings('ignore')

# Dependencias ML
!pip install -q timm albumentations
# Descarga de imágenes
!pip install -q duckduckgo-search requests pillow icrawler
# Exportación ONNX
!pip install -q onnx onnxruntime

import timm, PIL, onnx
print(f'✅ timm={timm.__version__}  PIL={PIL.__version__}  onnx={onnx.__version__}')
print('✅ Todas las dependencias instaladas.')

## CELDA 3 — Clonar repositorio

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import os, sys

REPO_URL    = 'https://github.com/nick2331/Electiva_3.git'
REPO_BRANCH = 'claude/analyze-project-tech-oKyKr'
REPO_DIR    = '/content/Electiva_3'

if not os.path.exists(REPO_DIR):
    !git clone --branch {REPO_BRANCH} --depth 1 {REPO_URL} {REPO_DIR}
    print('✅ Repositorio clonado.')
else:
    !git -C {REPO_DIR} pull origin {REPO_BRANCH} -q
    print('✅ Repositorio actualizado.')

os.chdir(REPO_DIR)
# Agregar raíz al sys.path para que los imports de ml.* funcionen
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

print(f'📁 CWD: {os.getcwd()}')
print(f'✅ sys.path incluye {REPO_DIR}')

## CELDA 4 — Descargar imágenes de vehículos

Usa **DuckDuckGo** (primario) + **icrawler/Bing** (respaldo) con:
- Validación pixel a pixel (rechaza siluetas, anime, logos)
- Deduplicación por hash MD5
- Reintentos automáticos si se bloquea

⏱️ ~25-35 minutos

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import time, hashlib, requests
from pathlib import Path
from io import BytesIO
from PIL import Image

OUTPUT_DIR = Path('/content/Electiva_3/ml/data/vehicleye_dataset')
VEHICLE_CLASSES = [
    ('Toyota','Corolla'),('Toyota','Hilux'),
    ('Chevrolet','Spark'),('Chevrolet','Aveo'),
    ('Renault','Logan'),('Renault','Sandero'),('Renault','Stepway'),
    ('Mazda','3'),('Mazda','CX-5'),
    ('Hyundai','Tucson'),('Hyundai','Accent'),
    ('Kia','Picanto'),('Kia','Rio'),
    ('Nissan','Frontier'),('Nissan','Versa'),
    ('Ford','Fiesta'),('Ford','Escape'),
    ('Volkswagen','Gol'),('Volkswagen','Jetta'),
    ('Suzuki','Swift'),
]

def to_dir(b, m):
    return f'{b}_{m}'.lower().replace('-','').replace(' ','_')

def is_valid(data):
    try:
        img = Image.open(BytesIO(data)).convert('RGB')
        w, h = img.size
        if w < 100 or h < 100: return False
        px = list(img.getdata()); n = len(px)
        black = sum(1 for r,g,b in px if r<40  and g<40  and b<40)
        white = sum(1 for r,g,b in px if r>220 and g>220 and b>220)
        gray  = sum(1 for r,g,b in px if abs(r-g)<12 and abs(g-b)<12)
        return black/n<0.55 and white/n<0.80 and gray/n<0.90
    except: return False

def download_class(brand, model, target=50):
    dname = to_dir(brand, model)
    cdir  = OUTPUT_DIR / dname
    cdir.mkdir(parents=True, exist_ok=True)

    existing = len(list(cdir.glob('*.jpg')))
    if existing >= target:
        print(f'  ✅ {brand} {model}: {existing} imgs (ya listo)')
        return existing

    from duckduckgo_search import DDGS
    queries = [
        f'{brand} {model} car exterior photo',
        f'{brand} {model} automobile 2020',
        f'{brand} {model} side view parked',
    ]
    done = existing; seen = set()
    for q in queries:
        if done >= target: break
        try:
            with DDGS() as ddgs:
                # timeout=10 en la sesión
                hits = list(ddgs.images(q, max_results=20, type_image='photo'))
        except: time.sleep(3); continue

        for h in hits:
            if done >= target: break
            try:
                # timeout=6 estricto para no colgar
                r = requests.get(h.get('image',''), timeout=6)
                if r.status_code != 200: continue
                md5 = hashlib.md5(r.content).hexdigest()
                if md5 in seen: continue
                seen.add(md5)
                if not is_valid(r.content): continue
                img = Image.open(BytesIO(r.content)).convert('RGB')
                img.save(cdir/f'img_{done:04d}.jpg','JPEG',quality=90)
                done += 1
            except: continue
        time.sleep(1)  # 1 seg entre queries (antes eran 2)

    n = len(list(cdir.glob('*.jpg')))
    print(f'  {"✅" if n>=20 else "⚠️ "} {brand:12} {model:10}: {n} imgs')
    return n

print('📥 Descargando (versión rápida)...\n')
total = 0
for b, m in VEHICLE_CLASSES:
    print(f'⬇️  {b} {m}...')
    total += download_class(b, m, target=50)

print(f'\n✅ Total: {total} imágenes')
print('Continúa con CELDA 5 → Entrenar' if total > 100 else '⚠️  Pocas imágenes, ejecuta de nuevo.')

## CELDA 5 — Entrenar EfficientNet-B0

⏱️ ~90 min con GPU T4. Puedes dejarlo corriendo.

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import os, sys

# Asegura que el proyecto esté en el path
REPO = '/content/Electiva_3'
os.chdir(REPO)
if REPO not in sys.path:
    sys.path.insert(0, REPO)

# Silencia warnings en el subprocess también
os.environ['PYTHONWARNINGS'] = 'ignore'
os.environ['PYTHONPATH']     = REPO

!python -W ignore ml/train.py \
    --data_dir ml/data/vehicleye_dataset \
    --epochs_phase1 5 \
    --epochs_phase2 10 \
    --output ml/checkpoints/efficientnet_b0_vehicleye.pth

from pathlib import Path
pth = Path('ml/checkpoints/efficientnet_b0_vehicleye.pth')
if pth.exists():
    print(f'\n✅ Modelo guardado: {pth.stat().st_size/1024/1024:.1f} MB')
    print('   Puedes continuar con CELDA 7 (exportar a ONNX).')
else:
    print('\n❌ No se generó el modelo. Revisa el error arriba.')

## CELDA 6 — Evaluar accuracy (opcional)

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import os; os.environ['PYTHONWARNINGS']='ignore'; os.environ['PYTHONPATH']='/content/Electiva_3'

!python -W ignore ml/evaluate.py

from pathlib import Path
f = Path('reports/model_metrics.txt')
if f.exists():
    print('\n📊 MÉTRICAS:\n' + '='*60)
    print(f.read_text())
else:
    print('⚠️  Sin reporte (normal si el dataset es pequeño).')

## CELDA 7 — Exportar a ONNX (formato Render)

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import io, os, sys, torch, timm, onnx
from pathlib import Path

REPO = '/content/Electiva_3'
os.chdir(REPO)
if REPO not in sys.path:
    sys.path.insert(0, REPO)

CHECKPOINT  = 'ml/checkpoints/efficientnet_b0_vehicleye.pth'
OUTPUT      = 'ml/checkpoints/vehicleye.onnx'
NUM_CLASSES = 20

pth = Path(CHECKPOINT)
if not pth.exists():
    raise FileNotFoundError('❌ No se encontró el checkpoint. Ejecuta CELDA 5 primero.')
print(f'✅ Checkpoint: {pth.stat().st_size/1024/1024:.1f} MB')

# Limpiar archivos ONNX previos (new PyTorch puede crear .onnx_data separado)
for old in Path(OUTPUT).parent.glob('vehicleye.onnx*'):
    old.unlink()
    print(f'🗑️  Eliminado: {old.name}')
Path(OUTPUT).parent.mkdir(parents=True, exist_ok=True)

model = timm.create_model('efficientnet_b0', pretrained=False, num_classes=NUM_CLASSES)
state = torch.load(CHECKPOINT, map_location='cpu', weights_only=True)
model.load_state_dict(state)
model.eval()
print('✅ Pesos cargados.')

dummy = torch.randn(1, 3, 224, 224)

# Exportar a BytesIO: fuerza el exportador legacy que embebe pesos inline
print('⏳ Exportando a buffer ONNX...')
buf = io.BytesIO()
with torch.no_grad(), warnings.catch_warnings():
    warnings.simplefilter('ignore')
    torch.onnx.export(
        model, dummy, buf,
        input_names=['input'],
        output_names=['logits'],
        export_params=True,
        do_constant_folding=True,
        opset_version=17,
    )

buf_mb = len(buf.getvalue()) / 1024 / 1024
print(f'📦 Buffer generado: {buf_mb:.1f} MB')

if buf_mb < 5:
    print('❌ Buffer muy pequeño — los pesos no se incluyeron en el export.')
    print('   Intentando con torch.jit.trace...')
    buf = io.BytesIO()
    with torch.no_grad(), warnings.catch_warnings():
        warnings.simplefilter('ignore')
        traced = torch.jit.trace(model, dummy)
        torch.onnx.export(
            traced, dummy, buf,
            input_names=['input'],
            output_names=['logits'],
            export_params=True,
            do_constant_folding=True,
            opset_version=17,
        )
    buf_mb = len(buf.getvalue()) / 1024 / 1024
    print(f'📦 Buffer (traced): {buf_mb:.1f} MB')

# Cargar con onnx y re-guardar con pesos embebidos (garantiza un solo archivo)
buf.seek(0)
onnx_model = onnx.load_model(buf)
onnx.save_model(onnx_model, OUTPUT, save_as_external_data=False)

size_mb = Path(OUTPUT).stat().st_size / 1024 / 1024
print(f'\n✅ ONNX guardado: {OUTPUT}')
print(f'   Tamaño final: {size_mb:.1f} MB')

# Verificar que no haya archivos de datos externos
extras = list(Path(OUTPUT).parent.glob('vehicleye.onnx*'))
print(f'   Archivos creados: {[e.name for e in extras]}')

if size_mb < 5:
    print('\n❌ Todavía muy pequeño — avisa a Claude con este mensaje completo.')
else:
    print('\n✅ Tamaño correcto — continúa con CELDA 8 para descargar.')


## CELDA 8 — Descargar modelo a tu computadora

In [ ]:
import warnings; warnings.filterwarnings('ignore')
from google.colab import files
from pathlib import Path

onnx = Path('ml/checkpoints/vehicleye.onnx')
pth  = Path('ml/checkpoints/efficientnet_b0_vehicleye.pth')

if onnx.exists():
    print(f'📥 Descargando vehicleye.onnx ({onnx.stat().st_size/1024/1024:.1f} MB)...')
    files.download(str(onnx))
    print('✅ Revisa tu carpeta Descargas.')
elif pth.exists():
    print('⚠️  vehicleye.onnx no existe. Descargando el .pth como respaldo...')
    files.download(str(pth))
    print('✅ Descargado efficientnet_b0_vehicleye.pth')
    print('   Ejecuta CELDA 7 para exportar a ONNX antes de subir a GitHub.')
else:
    print('❌ Ningún modelo encontrado. Ejecuta las celdas 5 y 7 primero.')

## CELDA 9 — Pasos siguientes

In [ ]:
print("""
🎉 ¡ENTRENAMIENTO COMPLETADO!

Tienes: vehicleye.onnx en tu carpeta Descargas.

══════════════════════════════════════════════════
PASO A — Subir a GitHub Releases
══════════════════════════════════════════════════
 1. github.com/nick2331/Electiva_3/releases
 2. "Create a new release"
 3. Tag: v1.0  |  Título: Modelo VehiclEye v1.0
 4. Arrastra vehicleye.onnx
 5. "Publish release"

══════════════════════════════════════════════════
PASO B — Copiar URL del modelo
══════════════════════════════════════════════════
 Clic derecho en vehicleye.onnx → Copiar enlace
 Ejemplo:
   https://github.com/nick2331/Electiva_3/
   releases/download/v1.0/vehicleye.onnx

══════════════════════════════════════════════════
PASO C — Configurar en Render
══════════════════════════════════════════════════
 dashboard.render.com → vehicleye-api → Environment
 Key:   MODEL_DOWNLOAD_URL
 Value: (URL del paso B)
 → Save  (redeploya automáticamente)

══════════════════════════════════════════════════
VERIFICACIÓN en Admin Panel
══════════════════════════════════════════════════
 Modelo IA: ✅ "Modelo ONNX cargado en memoria"

 ¡Las predicciones ahora son REALES! 🚗✅
""")